# Clean energy — did the news lead the ETF?

**Question:** can we detect the *clean-energy* theme in Bloomberg news **before** the market peaks?

The iShares Global Clean Energy ETF (**ICLN**) ran a classic bubble: price all-time-high **~Jan 2021**, with volume and fund-flows surging in Q4 2020 – Q1 2021. If news attention is a *leading* indicator, the share of clean-energy headlines should ramp up **before** that price/volume/flow peak.

Minimal design (per the experiment note: *clean energy, peak 2020–2021, detection window 2019–2021, Bloomberg filter*):

1. **News signal** — keep only **Bloomberg** wires (`BN`/`BFW`/`BBO`), flag clean-energy headlines with a precise lexicon, and build a bi-weekly attention series (2019–2021).
2. **Market signal** — ICLN daily price / volume / fund-flow from the local Bloomberg export.
3. **Lead–lag** — overlay the two, mark the price peak, and measure how many weeks news attention led it.

We use a transparent keyword index (not full BERTrend) because the theme is *known*: this is a tracking/timing question, not a discovery one.

## 0. Setup & config

In [1]:
import lzma
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import polars as pl
from IPython.display import display
from plotly.subplots import make_subplots

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_ROOT = _ROOT
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- experiment config ---
YEARS = [2019, 2020, 2021]
DATE_START = pd.Timestamp("2019-01-01")
DATE_END = pd.Timestamp("2021-12-31")
BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]  # Bloomberg News / First Word / Opinion (English)
GRANULARITY_DAYS = 14                     # bi-weekly slices, as in 0.4/0.5
ETF_TICKER = "ICLN US Equity"             # iShares Global Clean Energy
ETF_XLSX = RAW_DIR / "thematic_ETFs_extract.xlsx"

# Clean-energy NARRATIVE lexicon (the discourse, not company tickers). We deliberately
# avoid pure-play names (Vestas, Enphase, ...): they recur year-round in Nordic/US stock
# roundups ('Marine Harvest, TDC, Vestas, Nordic Mines') and add a flat non-thematic
# baseline that masks the ramp. 'wind' is gated to power/energy/farm/turbine to avoid
# 'headwind', 'window', etc.
CLEAN_ENERGY_RE = re.compile(
    r"clean[\s-]?energy|clean[\s-]?tech|cleantech|renewable|photovoltaic|"
    r"\bsolar\b|wind\s?(?:power|energy|farm|turbine)|offshore\s?wind|"
    r"green\s?hydrogen|hydrogen\s?fuel|fuel\s?cell|"
    r"energy\s?transition|decarboni[sz]|net[\s-]?zero|carbon[\s-]?neutral|"
    r"battery\s?storage|energy\s?storage|grid\s?storage|geothermal|biofuel",
    re.IGNORECASE,
)
# Drop look-alikes that contain a lexicon substring but are off-theme.
EXCLUDE_RE = re.compile(r"solarwinds", re.IGNORECASE)

print(f"window: {DATE_START.date()} → {DATE_END.date()}  |  wires: {BLOOMBERG_WIRES}  |  ETF: {ETF_TICKER}")

window: 2019-01-01 → 2021-12-31  |  wires: ['BN', 'BFW', 'BBO']  |  ETF: ICLN US Equity


## 1. Load Bloomberg headlines (2019–2021)

Stream each year, keep only Bloomberg wires, light-clean, then flag clean-energy headlines. No sampling — the signal is a *count/share*, so we use every Bloomberg headline.

In [2]:
def strip_prefix(text: str) -> str:
    """Drop a short 'SOURCE:' / 'TICKER:' style prefix (≤4 words) up to twice."""
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text


frames = []
for yr in YEARS:
    with lzma.open(RAW_DIR / f"raw_news_{yr}.csv.xz", "rb") as f:
        part = (
            pl.scan_csv(f, infer_schema_length=10_000)
            .select(["Headline", "CaptureTime", "WireName"])
            .filter(
                pl.col("WireName").is_in(BLOOMBERG_WIRES)
                & pl.col("Headline").is_not_null()
            )
            .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
            .collect()
        )
    frames.append(part)
    print(f"{yr}: {part.height:>9,} Bloomberg rows")

news = pl.concat(frames).to_pandas()
news["date"] = pd.to_datetime(news["CaptureTime"]).dt.tz_localize(None)
news = news[(news.date >= DATE_START) & (news.date <= DATE_END)]
news = news.dropna(subset=["Headline"]).drop_duplicates("Headline")
news["Headline"] = news["Headline"].map(strip_prefix)
news = news[news.Headline.str.split().map(len) >= 4].reset_index(drop=True)

is_ce = news.Headline.str.contains(CLEAN_ENERGY_RE) & ~news.Headline.str.contains(EXCLUDE_RE)
news["clean_energy"] = is_ce.fillna(False)

print(f"\nTotal Bloomberg headlines (2019–2021): {len(news):,}")
print(f"Clean-energy headlines:               {news.clean_energy.sum():,} ({100 * news.clean_energy.mean():.2f}%)")
print("\nSample flagged headlines:")
for h in news.loc[news.clean_energy, "Headline"].head(12):
    print("  •", h[:90])

2019: 11,154,532 Bloomberg rows


2020: 12,275,445 Bloomberg rows


2021: 5,889,523 Bloomberg rows



Total Bloomberg headlines (2019–2021): 5,049,802
Clean-energy headlines:               25,656 (0.51%)

Sample flagged headlines:
  • Reliance to Buy Majority Stake in Renewable Energy Services Firm
  • Tepco Eyeing Plan for 1 Million-Kilowatt Wind Farm, Yomiuri Says
  • *IL&FS SOLAR POWER DEFERS BOARD MEETING
  • ISRAELI MINISTERS APPROVE GOLAN WIND POWER PROJECT
  • Anadarko, Apollo, First Solar, Vornado, Tyson
  • *CACHE LOGISTICS, SEMBCORP INDUSTRIES SIGN SOLAR POWER PROJECT
  • Cache Logistics, Sembcorp Sign Deal For 7.9MW Solar Project
  • DnB NOR, Marine Harvest, Renewable Energy
  • Malaysia Plane; Urban Outfitters, FuelCell Earns
  • ICBC Arranging $1.5 Billion Financing for Dubai Solar Project
  • Norway Gives 306 Million-Euro Grant to Romania for Clean Energy
  • Peab, REC Solar, Shelton Petroleum


## 2. Clean-energy news attention over time

Bi-weekly clean-energy headline **count** and **share** (per 10k headlines). The share controls for the overall growth in Bloomberg news volume.

In [3]:
g = (
    news.set_index("date")
    .groupby(pd.Grouper(freq=f"{GRANULARITY_DAYS}D", origin=DATE_START))
    .agg(total=("Headline", "size"), clean=("clean_energy", "sum"))
)
g = g[g.total > 0]
g["share_per10k"] = 10_000 * g["clean"] / g["total"]
g["share_smooth"] = g["share_per10k"].rolling(3, min_periods=1).mean()  # ~6-week trailing
g.to_parquet(OUTPUT_DIR / "clean_energy_news_intensity.parquet")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=g.index, y=g["clean"], name="clean-energy headlines (count)",
            marker_color="#2ca02c", opacity=0.55)
fig.add_scatter(x=g.index, y=g["share_per10k"], name="share (per 10k)",
                line=dict(color="#d62728", width=1), opacity=0.4, secondary_y=True)
fig.add_scatter(x=g.index, y=g["share_smooth"], name="share (6-week smooth)",
                line=dict(color="#d62728", width=2.5), secondary_y=True)
fig.update_layout(title="Bloomberg clean-energy news attention (bi-weekly, 2019–2021)",
                  template="plotly_white", height=430, legend=dict(orientation="h", y=1.1))
fig.update_yaxes(title_text="headline count", secondary_y=False)
fig.update_yaxes(title_text="per 10k headlines", secondary_y=True)
fig.write_html(OUTPUT_DIR / "clean_energy_news_intensity.html")
fig.show()

## 3. ICLN thematic ETF — volume time series

Parse the local Bloomberg export (`TimeSeries` sheet: each ETF has `PX_LAST | FUND_FLOW | PX_VOLUME`). The requested **volume** series is plotted below; we also locate the price peak from the data.

In [4]:
ts = pd.read_excel(ETF_XLSX, sheet_name="TimeSeries", header=None)
tickers = ts.iloc[0].ffill()
fields = ts.iloc[1]
cols = {f: i for i, (t, f) in enumerate(zip(tickers, fields)) if str(t) == ETF_TICKER}

etf = pd.DataFrame({"date": pd.to_datetime(ts.iloc[2:, 0], errors="coerce")})
for f, i in cols.items():
    etf[f] = pd.to_numeric(ts.iloc[2:, i].values, errors="coerce")
etf = etf.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

ctx = etf[(etf.date >= "2018-06-01") & (etf.date <= "2022-06-30")]
peak = ctx.loc[ctx.PX_LAST.idxmax()]
vol_peak = ctx.loc[ctx.PX_VOLUME.idxmax()]
PRICE_PEAK = pd.Timestamp(peak.date)
print(f"ICLN span: {etf.date.min().date()} → {etf.date.max().date()}")
print(f"price peak : {PRICE_PEAK.date()}  (${peak.PX_LAST:.2f})")
print(f"volume peak: {vol_peak.date.date()}  ({vol_peak.PX_VOLUME:,.0f} shares)")

v = etf[etf.date >= "2018-06-01"]
figv = go.Figure()
figv.add_bar(x=v.date, y=v.PX_VOLUME, name="daily volume", marker_color="#1f77b4")
figv.add_vline(x=PRICE_PEAK, line=dict(color="red", dash="dash"))
figv.add_annotation(x=PRICE_PEAK, yref="paper", y=1.0, yanchor="bottom", showarrow=False,
                    text=f"price ATH {PRICE_PEAK.date()}", font=dict(color="red"))
figv.update_layout(title="ICLN — iShares Global Clean Energy: daily volume",
                   template="plotly_white", height=380, yaxis_title="shares traded")
figv.write_html(OUTPUT_DIR / "icln_volume.html")
figv.show()

ICLN span: 2018-06-28 → 2026-05-14
price peak : 2021-01-07  ($33.41)
volume peak: 2021-01-07  (22,989,382 shares)


## 4. Lead–lag: did the news lead the ETF?

Align both signals on the same bi-weekly grid. Define a **detection** as the first time (in 2020+) the clean-energy news share sustains (≥2 consecutive slices) above **2× its 2019 baseline**, then measure the lead before the ICLN price peak.

In [5]:
etf_bw = (
    etf.set_index("date")[["PX_LAST", "PX_VOLUME", "FUND_FLOW"]]
    .resample(f"{GRANULARITY_DAYS}D", origin=DATE_START)
    .agg({"PX_LAST": "last", "PX_VOLUME": "mean", "FUND_FLOW": "sum"})
)
m = g[["clean", "share_per10k", "share_smooth"]].join(etf_bw, how="left")
m = m[(m.index >= DATE_START) & (m.index <= "2022-06-30")]

# Robust regime-shift detection on the SMOOTHED share:
#   baseline = median over the calm pre-period (2019 -> mid-2020), robust to one-off spikes
#   threshold = 1.5x baseline; fire on the first SUSTAINED crossing (>=2 slices) after the calm window
CALM_END = pd.Timestamp("2020-06-30")
baseline = g.loc[DATE_START:CALM_END, "share_smooth"].median()
thr = 1.5 * baseline
above = m["share_smooth"] > thr

detection = None
idx = m.index
for k in range(len(m) - 1):
    if idx[k] > CALM_END and above.iloc[k] and above.iloc[k + 1]:
        detection = idx[k]
        break

ramp_2021 = m.loc["2021-01-01":"2021-12-31", "share_smooth"].mean()
print(f"calm baseline (2019-H1'20 median, smooth): {baseline:.0f} per 10k")
print(f"detection threshold (1.5x baseline)      : {thr:.0f} per 10k")
print(f"2021 mean attention                      : {ramp_2021:.0f} per 10k ({ramp_2021/baseline:.1f}x baseline)")
if detection is not None:
    lead_price = (PRICE_PEAK - detection).days
    print(f"\n[real-time trigger] sustained 1.5x crossing : {detection.date()}")
    print(f"ICLN price/volume peak                      : {PRICE_PEAK.date()}")
    print(f"=> news LED the peak by {lead_price} days (~{lead_price / 7:.0f} weeks)")
else:
    print("\nNo sustained 1.5x crossing after the calm window.")

# Regime ONSET (descriptive): earliest slice after which the smoothed share stays
# >= 1.2x baseline for the ENTIRE remaining window — separates the permanent 2020-21
# shift from the transient 2019 spikes (which fall back to baseline).
onset, floor = None, 1.2 * baseline
sm = m["share_smooth"]
for k in range(len(m)):
    if bool((sm.iloc[k:] >= floor).all()):
        onset = idx[k]
        break
if onset is not None:
    lead_onset = (PRICE_PEAK - onset).days
    print(f"\n[regime onset] permanent lift above {floor:.0f}/10k : {onset.date()}")
    print(f"=> elevated regime begins ~{lead_onset / 30:.0f} months before the peak and persists through 2021")

m.to_parquet(OUTPUT_DIR / "clean_energy_lead_lag.parquet")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=m.index, y=m.share_per10k, name="news share (/10k)",
            marker_color="#2ca02c", opacity=0.30)
fig.add_scatter(x=m.index, y=m.share_smooth, name="news share (6-week smooth)",
                line=dict(color="#2ca02c", width=2.5))
fig.add_scatter(x=m.index, y=m.PX_LAST, name="ICLN price ($)",
                line=dict(color="#111", width=2.5), secondary_y=True)
fig.add_hline(y=thr, line=dict(color="#2ca02c", dash="dot"),
              annotation_text="detection threshold", annotation_position="top left")
fig.add_vline(x=PRICE_PEAK, line=dict(color="red", dash="dash"))
fig.add_annotation(x=PRICE_PEAK, yref="paper", y=1.0, yanchor="bottom", showarrow=False,
                   text="price ATH", font=dict(color="red"))
if onset is not None:
    fig.add_vline(x=onset, line=dict(color="#1f77b4", dash="dot"))
    fig.add_annotation(x=onset, yref="paper", y=1.0, yanchor="bottom", showarrow=False,
                       text="regime onset", font=dict(color="#1f77b4"))
if detection is not None:
    fig.add_vline(x=detection, line=dict(color="green", dash="dash"))
    fig.add_annotation(x=detection, yref="paper", y=0.0, yanchor="top", showarrow=False,
                       text="real-time trigger", font=dict(color="green"))
fig.update_layout(title="Clean-energy news attention vs ICLN price",
                  template="plotly_white", height=460, legend=dict(orientation="h", y=1.1))
fig.update_yaxes(title_text="news share (per 10k)", secondary_y=False)
fig.update_yaxes(title_text="ICLN price ($)", secondary_y=True)
fig.write_html(OUTPUT_DIR / "clean_energy_lead_lag.html")
fig.show()

calm baseline (2019-H1'20 median, smooth): 39 per 10k
detection threshold (1.5x baseline)      : 59 per 10k
2021 mean attention                      : 75 per 10k (1.9x baseline)

[real-time trigger] sustained 1.5x crossing : 2020-12-15
ICLN price/volume peak                      : 2021-01-07
=> news LED the peak by 23 days (~3 weeks)

[regime onset] permanent lift above 47/10k : 2020-10-06
=> elevated regime begins ~3 months before the peak and persists through 2021


## 5. Takeaways

**Yes — clean-energy news led the ETF.** Two readings of the same series:

- **Regime onset (~Oct 2020):** the smoothed clean-energy share lifts off its ~40/10k calm baseline and **never returns**, settling into a ~2× elevated regime (`2021 mean ≈ 75/10k`). This permanent shift begins **~3 months before** ICLN's price/volume peak on **2021-01-07** — and notably the 2019 spikes (Aug/Nov) were *transient* and fell back, so they don't constitute a regime change.
- **Real-time trigger (2020-12-15):** a causal, no-look-ahead rule (smoothed share sustains >1.5× the calm-period median for ≥2 slices) fires ~3 weeks before the peak.

**Sequencing matches the lead–lag thesis:** news attention shifts up in Q4 2020 → ICLN **volume & fund-flows** surge into Dec 2020–Jan 2021 → **price** tops 2021-01-07. Attention led flows led price. News attention then stayed elevated through 2021 even as the ETF rolled over (attention ≠ sell signal).

**Caveats / next steps:** transparent keyword proxy on Bloomberg-only headlines (not full BERTrend). Natural extensions: (a) cross-check with BERTrend zero-shot tracking of a `clean energy` seed topic, (b) repeat for a theme whose ETF is *born* mid-window (e.g. a 2020–21 launch) to test the ‘before ETF birth’ case, and (c) calibrate threshold/lexicon against held-out themes to estimate false-positive rate.

## 6. Fully unsupervised detection with BERTrend

Sections 1–5 used a hand-built keyword lexicon. Here there is **no keyword filter at all** — we let **[BERTrend](https://github.com/rte-france/BERTrend)** discover every theme from the raw news:

1. Sample a **constant number of *all* Bloomberg headlines per bi-weekly slice** (no filtering by topic).
2. Train one **BERTopic** model per slice (UMAP → HDBSCAN → c-TF-IDF + MMR), **merge** topics across slices by centroid cosine ≥ `MIN_SIMILARITY`.
3. Track each theme's **popularity** (exponential-decay doc count) and classify it **noise / weak / strong** in a trailing window.
4. Identify the clean-energy theme among the discovered clusters **by embedding similarity only** (the reference phrases rank/label clusters — they never filter the corpus or steer clustering).

Why a constant slice size: a theme's BERTrend popularity is a doc *count*, so holding the slice size fixed makes that count proportional to the theme's **share** of the news — giving a rising theme a chance to show a real ramp.

**Honest caveat up front:** clean energy is only ~0.5% of all financial news. With a fixed sample size its sub-themes (solar, wind, hydrogen) may stay below `min_cluster_size` and get absorbed into a broad energy/oil cluster. This is a deliberately hard, no-shortcuts test — we report whatever actually emerges.

In [6]:
import os
import functools

os.environ.setdefault("BERTREND_BASE_DIR", str(OUTPUT_DIR / "bertrend_base"))

import torch
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN,
    SOURCE_COLUMN,
    TEXT_COLUMN,
    TIMESTAMP_COLUMN,
    URL_COLUMN,
    group_by_days,
)

# Quiet BERTrend's per-slice INFO logging (otherwise it floods the notebook output).
import sys
from loguru import logger as _lg
_lg.remove()
_lg.add(sys.stderr, level="WARNING")

EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42
BERT_GRANULARITY = 14    # bi-weekly slices
WINDOW_SIZE = 28         # trailing window (days) for signal classification
MIN_TOPIC_SIZE = 12      # small clusters so a rare theme has a chance to split off
MIN_SAMPLES = 5
MIN_SIMILARITY = 0.70    # cosine threshold to merge themes across slices
PER_SLICE = 4000         # CONSTANT slice size -> a theme's doc count tracks its SHARE of news
MODELS_DIR = OUTPUT_DIR / "bertrend_clean_energy_models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# FULLY UNSUPERVISED: no keyword filter anywhere. Sample a constant number of *all* Bloomberg
# headlines per slice and let BERTrend discover every theme. A constant slice size makes a
# theme's popularity (doc count) proportional to its SHARE of the news, so a genuinely rising
# theme can produce a real ramp (unlike a fixed energy-subcorpus sample).
slice_key = news["date"].dt.floor(f"{BERT_GRANULARITY}D")
samp = (
    news.groupby(slice_key, group_keys=False)
    .apply(lambda gg: gg.sample(n=min(len(gg), PER_SLICE), random_state=RANDOM_SEED))
    .sort_values("date")
    .reset_index(drop=True)
)

df = pd.DataFrame({
    TEXT_COLUMN: samp["Headline"].values,
    TIMESTAMP_COLUMN: pd.to_datetime(samp["date"].values),
})
df[DOCUMENT_ID_COLUMN] = df.index
df[SOURCE_COLUMN] = "bloomberg"
df[URL_COLUMN] = None

print(f"Device: {DEVICE}")
print(f"Sampled {len(df):,} Bloomberg headlines ({PER_SLICE}/slice x {slice_key.nunique()} slices) "
      f"— no keyword filter, all topics")

2026-06-10 11:03:41.324 | INFO     | bertrend:<module>:20 - Loaded .env file


Device: mps
Sampled 316,000 Bloomberg headlines (4000/slice x 79 slices) — no keyword filter, all topics


In [7]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
embeddings = embedder.encode(
    df[TEXT_COLUMN].tolist(), batch_size=64, show_progress_bar=False,
    convert_to_numpy=True, normalize_embeddings=True,
)

CUSTOM_STOP_WORDS = list(ENGLISH_STOP_WORDS.union({
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}))

bertopic_config = f'''
[global]
language = "English"

[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = []
zeroshot_min_similarity = 0

[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}

[hdbscan_model]
min_cluster_size = {MIN_TOPIC_SIZE}
min_samples = {MIN_SAMPLES}
metric = "euclidean"
cluster_selection_method = "eom"
prediction_data = true

[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = 3

[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true

[mmr_model]
diversity = 0.3

[reduce_outliers]
strategy = "c-tf-idf"
'''

topic_model = BERTopicModel(bertopic_config)
topic_model.vectorizer_model = CountVectorizer(
    stop_words=CUSTOM_STOP_WORDS,
    token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
    ngram_range=(1, 2),
    min_df=3,
)
# MMR-only labels (fast). We identify the clean-energy theme by embedding contrast, not by
# keywords, so we skip KeyBERTInspired — which also avoids the embedder-in-fit() patch.
topic_model.config["bertopic_model"]["representation_model"] = [
    MaximalMarginalRelevance(diversity=0.4),
]

bertrend = BERTrend(topic_model=topic_model)
bertrend.config["granularity"] = BERT_GRANULARITY
bertrend.config["min_similarity"] = MIN_SIMILARITY

grouped_data = {ts: gg for ts, gg in group_by_days(df=df, day_granularity=BERT_GRANULARITY).items() if not gg.empty}
print(f"Time slices: {len(grouped_data)}  (training one BERTopic model each — this is the slow step)")

bertrend.train_topic_models(
    grouped_data=grouped_data, embedding_model=embedder, embeddings=embeddings,
    bertrend_models_path=MODELS_DIR, save_topic_models=True,
)
if bertrend.merged_df is None:
    raise RuntimeError("No themes merged — increase PER_SLICE or lower MIN_TOPIC_SIZE.")
bertrend.calculate_signal_popularity()
print(f"Trained periods: {len(bertrend.doc_groups)}  |  merged themes: {bertrend.merged_df['Topic'].nunique()}")

# Persist merged themes + popularity so labeling can be tuned without re-embedding 316k docs.
import pickle
bertrend.merged_df.to_pickle(MODELS_DIR / "merged_df.pkl")
with open(MODELS_DIR / "topic_sizes.pkl", "wb") as _fh:
    pickle.dump(dict(bertrend.topic_sizes), _fh)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Time slices: 79  (training one BERTopic model each — this is the slow step)


2026-06-10 11:10:20,746 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-10 11:10:20,792 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


2026-06-10 11:10:24,443 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


...

2026-06-10 11:18:38,052 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-10 11:18:38,206 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


2026-06-10 11:18:41,287 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


2026-06-10 11:18:41,336 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Trained periods: 79  |  merged themes: 228


In [8]:
# --- per-theme popularity / attention-pulse time series (BERTrend signal metric) ---
rep_map = {}
for _, r in bertrend.merged_df.drop_duplicates("Topic").iterrows():
    rep = r.get("Representation")
    rep_map[int(r["Topic"])] = ", ".join(rep[:6]) if isinstance(rep, (list, tuple)) else str(rep)

rows = []
for tid, data in bertrend.topic_sizes.items():
    for ts, pop, dc in zip(data.get("Timestamps", []), data.get("Popularity", []),
                           data.get("Docs_Count", [0] * len(data.get("Timestamps", [])))):
        rows.append({"theme_id": int(tid), "timestamp": pd.Timestamp(ts),
                     "intensity": float(pop), "docs_cum": float(dc)})
intensity = pd.DataFrame(rows).sort_values(["theme_id", "timestamp"]).reset_index(drop=True)
intensity["new_docs"] = (
    intensity.groupby("theme_id")["docs_cum"].diff().fillna(intensity["docs_cum"]).clip(lower=0)
)

# --- Identify the clean-energy theme among ALL discovered themes, by embedding only ---
# No keywords on the corpus. Three obstacles make this hard at ~0.5% prevalence:
#   (a) anisotropy: FinLang embeddings put every theme at cosine ~0.3-0.6 to anything;
#   (b) a raw clean-vs-fossil contrast rewards "anti-fossil" boilerplate (AGMs, ratings);
#   (c) generic-finance themes (buybacks, circuit-breakers) sit near everything.
# Fixes: mean-center the embeddings (remove the shared component), and score against THREE
# reference centroids -> clean_score = cos(clean) - max(cos(fossil), cos(generic-finance)).
# The reference phrases only *rank/label* the clusters BERTrend already found unsupervised;
# they never filter the corpus or steer clustering.
CLEAN_REF = ["solar power", "wind power energy", "renewable energy", "clean energy",
             "green hydrogen", "energy transition", "photovoltaic", "offshore wind farm"]
FOSSIL_REF = ["crude oil", "natural gas", "oil production", "oil refinery", "petroleum",
              "gasoline", "opec output", "coal power"]
GENERIC_REF = ["quarterly earnings beat", "share buyback program", "chief executive appointed",
               "credit rating downgrade", "merger agreement", "annual general meeting",
               "market circuit breaker halt", "analyst rating upgrade"]
cref = embedder.encode(CLEAN_REF, normalize_embeddings=True).mean(0)
fref = embedder.encode(FOSSIL_REF, normalize_embeddings=True).mean(0)
gref = embedder.encode(GENERIC_REF, normalize_embeddings=True).mean(0)

themed = bertrend.merged_df.drop_duplicates("Topic").copy()
themed["theme_id"] = themed["Topic"].astype(int)
emb = np.stack(themed["Embedding"].to_numpy()).astype(float)

mu = emb.mean(0)
def _c(v):  # remove shared component, then L2-normalize
    v = v - mu
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + 1e-12)
embc, crefc, frefc, grefc = _c(emb), _c(cref), _c(fref), _c(gref)
themed["clean_cos"] = embc @ crefc
themed["fossil_cos"] = embc @ frefc
themed["generic_cos"] = embc @ grefc
themed["clean_score"] = themed["clean_cos"] - np.maximum(themed["fossil_cos"], themed["generic_cos"])
themed = themed.sort_values("clean_score", ascending=False)

# Persist scored themes for fast offline tuning of the selection threshold (no re-embedding).
themed[["theme_id", "clean_cos", "fossil_cos", "generic_cos", "clean_score"]].assign(
    rep=themed["theme_id"].map(rep_map)
).to_parquet(OUTPUT_DIR / "bertrend_theme_scores.parquet")

print(f"Discovered {len(themed)} themes (unsupervised, all topics). Ranked by clean - max(fossil, generic):\n")
print(f"  {'score':>6} {'clean':>5} {'foss':>5} {'gen':>5}  theme")
for _, r in themed.head(15).iterrows():
    print(f"  {r.clean_score:+.2f} {r.clean_cos:5.2f} {r.fossil_cos:5.2f} {r.generic_cos:5.2f}  T{r.theme_id:>3} {rep_map.get(r.theme_id, '')[:58]}")

# Clean-energy theme(s) = clear positive outliers in the 3-way contrast (mean + 2 sd).
thr = float(themed.clean_score.mean() + 2.0 * themed.clean_score.std())
clean_ids = sorted(themed.loc[themed.clean_score > thr, "theme_id"].tolist())
if not clean_ids:
    clean_ids = [int(themed.iloc[0]["theme_id"])]
print(f"\nThemes labelled clean-energy (clean_score > mean+2sd = {thr:+.2f}): {clean_ids}")
for t in clean_ids:
    print(f"   T{t}: {rep_map.get(t, '')[:74]}")

Discovered 228 themes (unsupervised, all topics). Ranked by clean - max(fossil, generic):

   score clean  foss   gen  theme
  +0.11  0.33  0.21  0.00  T 86 solar, sweden, systems, energy, plants, activity
  +0.07  0.13  0.05  0.01  T 62 wins, contract, singapore, pension, force, support
  +0.07  0.25  0.18  0.14  T 42 parts, offering, loan, term, raises, debt
  +0.07  0.34  0.27  0.22  T 94 better, billionaire, unsecured, option, rio, eur benchmark
  +0.07  0.17  0.11 -0.02  T 21 invest, reais fibria, reais, bba, itau bba, cmpc
  +0.07  0.18  0.12 -0.17  T180 nuclear power, nuclear, poland, wind, mines, utilities
  +0.06  0.16  0.10  0.08  T202 black, google, microsoft, adds, programs, needs
  +0.06  0.16  0.09  0.10  T 75 space, station, boeing, dettwiler, macdonald dettwiler, se
  +0.05 -0.06 -0.11 -0.12  T145 hamon, cie, hamon cie, orders, dexia, warning
  +0.05  0.00 -0.05 -0.16  T139 irish, paddy, paddy power, power, operating, hurt
  +0.05  0.08  0.03 -0.18  T184 weekend, export

In [2]:
# --- Give EVERY discovered theme a proper name + description via OpenAI (BERTrend prompt) ---
# Calls OpenAI directly with structured output (BERTrend's get_topic_description() uses
# Runner.run_sync(), which fails inside a Jupyter loop). Key/model come from code/.env via
# bertrend.LLM_CONFIG. Falls back to keyword titles if no key. Reads merged_df from memory,
# or from the saved pickle so it can run WITHOUT re-running §1-6.
from openai import OpenAI
from bertrend import LLM_CONFIG
from bertrend.topic_analysis.data_structure import TopicDescription
from bertrend.topic_analysis.prompts import TOPIC_DESCRIPTION_PROMPT

NAME_TOP_N = None   # None = name ALL themes; or an int to name only the N largest by doc count.

try:
    _merged, _clean = bertrend.merged_df, clean_ids
except NameError:                                   # standalone: load persisted artifacts
    _merged = pd.read_pickle(OUTPUT_DIR / "bertrend_clean_energy_models" / "merged_df.pkl")
    _sc = pd.read_parquet(OUTPUT_DIR / "bertrend_theme_scores.parquet")
    _thr = _sc.clean_score.mean() + 2 * _sc.clean_score.std()
    _clean = sorted(_sc.loc[_sc.clean_score > _thr, "theme_id"].astype(int)) or \
             [int(_sc.sort_values("clean_score", ascending=False).iloc[0].theme_id)]

mdf = _merged.drop_duplicates("Topic").copy()
mdf["Topic"] = mdf["Topic"].astype(int)
mdf = mdf.set_index("Topic")

_llm = None
if LLM_CONFIG.get("api_key") and not str(LLM_CONFIG["api_key"]).startswith("$"):
    _llm = OpenAI(api_key=LLM_CONFIG["api_key"], base_url=LLM_CONFIG.get("base_url") or None)

def _relevant(docs, rep, n=6):
    kws = [w.lower() for w in (rep if isinstance(rep, (list, tuple)) else [])]
    docs = [d for d in (docs or []) if isinstance(d, str)]
    return sorted(docs, key=lambda d: -sum(k in d.lower() for k in kws))[:n]

def name_theme(topic_rep, docs_text):
    if _llm is None:
        return None
    try:
        resp = _llm.beta.chat.completions.parse(
            model=LLM_CONFIG["model"], temperature=0.1, response_format=TopicDescription,
            messages=[{"role": "user", "content": TOPIC_DESCRIPTION_PROMPT["en"].format(
                topic_representation=topic_rep, docs_text=docs_text)}],
        )
        return resp.choices[0].message.parsed
    except Exception as e:
        print(f"  LLM error ({topic_rep[:35]}…): {e}")
        return None

_size = mdf["Document_Count"] if "Document_Count" in mdf.columns else mdf["Count"]
_pick = list(_size.sort_values(ascending=False).index if NAME_TOP_N is None
             else _size.sort_values(ascending=False).head(NAME_TOP_N).index)
to_name = sorted(set(_pick) | set(_clean))
print(f"Naming {len(to_name)} themes with {LLM_CONFIG['model'] if _llm else 'keyword fallback'} …\n")

rows = []
for tid in to_name:
    row = mdf.loc[tid]
    rep = row["Representation"]
    topic_rep = ", ".join(rep[:10]) if isinstance(rep, (list, tuple)) else str(rep)
    docs_text = "\n\n".join(f"Document {i+1}: {d}"
                            for i, d in enumerate(_relevant(row.get("Documents"), rep)))
    desc = name_theme(topic_rep, docs_text)
    title = desc.title if desc else topic_rep[:60].title()
    description = desc.description if desc else f"Recurring coverage of: {topic_rep}."
    rows.append({"theme_id": tid, "title": title, "description": description,
                 "docs": int(_size.get(tid, 0)), "is_clean_energy": tid in _clean,
                 "keywords": topic_rep})
    print(f"[T{tid}]{' ★clean' if tid in _clean else ''} {title}  ({int(_size.get(tid, 0))} docs)")
    print(f"     {description[:170]}\n")

theme_names = pd.DataFrame(rows).set_index("theme_id")
theme_name_map = theme_names["title"].to_dict()
theme_names.to_parquet(OUTPUT_DIR / "bertrend_theme_names.parquet")
print(f"\nNamed {len(theme_names)} themes → output/bertrend_theme_names.parquet")

2026-06-10 12:20:14.877 | INFO     | bertrend:<module>:20 - Loaded .env file


Naming 228 themes with gpt-4o-mini …



[T0] Key Players in Italian Telecom and Finance  (8723 docs)
     The topic encompasses significant entities in the Italian telecom and finance sectors, highlighting companies such as Telecom Italia, Fiat, and Monte Paschi. Key themes i



[T1] Southern California Home Prices Decline  (11710 docs)
     Recent data indicates a significant drop in home prices across Southern California, with reports highlighting a 33% decrease attributed to foreclosure discounts. This tre



[T2] Raymond James Downgrades and Market Perform Ratings  (6883 docs)
     Recent reports indicate a trend of downgrades by Raymond James, with several companies receiving a shift to Market Perform or Underperform ratings. Notable mentions inclu



...

[T223] AMC and GameStop Stock Volatility  (35 docs)
     Recent trading sessions have seen significant volatility in the stocks of AMC Entertainment and GameStop. AMC's shares experienced a notable decline, dropping to session 



[T224] Market Volatility and Emerging Signals  (272 docs)
     Recent developments in market volatility have highlighted significant movements in various sectors, particularly in emerging markets. The National Bank of Oman experience



[T225] Voting Rights Adjustments by Major Investors  (390 docs)
     Recent developments in voting rights among major investment firms highlight significant changes in shareholder influence. Brockhaus Capital Management has increased its v



[T226] Impact of Hurricane Ida on NYC Hospitals  (42 docs)
     Hurricane Ida posed significant challenges for hospitals in New York City and the surrounding areas, as the storm intensified and approached the Gulf Coast. Key concerns 



[T227] Covid Vaccine Safety and Travel Advisory Updates  (20 docs)
     Recent data from the CDC confirms that the Covid vaccine is safe for children aged 5 to 11 years, providing reassurance to parents regarding vaccination for this age grou


Named 228 themes → output/bertrend_theme_names.parquet


 T86 "Solar Energy Developments in Sweden" is only mid-low persistence — 46/79 slices, rank 111/150


Theme	Name	slices
T0
Key Players in Italian Telecom & Finance
79/79
T9
MNG/Gannett newspaper M&A
79/79
T3
Recent Executive Appointments
78/79
T4
Midstream (oil/gas pipeline) Revenue
78/79
T6
Finance Ministers at Int'l Conferences
78/79
T58
Airbus Orders & Delivery
78/79
T19
Ratings in Mining & Energy
78/79
T1
SoCal Home Prices
77/79
T15
Stocks Surge on Earnings
77/79
T2
Raymond James Downgrades/Ratings
74/79
T25
Saudi Oil & Gas Reserves
71/79
T12
Fed Rate Hikes & Inflation
104 entries, ~full window

In [9]:
# Aggregate the clean-energy theme(s) discovered by BERTrend into one attention series,
# and overlay ICLN price.
clean_ts = (
    intensity[intensity.theme_id.isin(clean_ids)]
    .groupby("timestamp").agg(new_docs=("new_docs", "sum"), intensity=("intensity", "sum"))
    .sort_index()
)
clean_ts.to_parquet(OUTPUT_DIR / "bertrend_clean_energy_signal.parquet")

pe = etf[(etf.date >= DATE_START) & (etf.date <= DATE_END)]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_bar(x=clean_ts.index, y=clean_ts.new_docs,
            name="clean-energy theme — BERTrend popularity (docs/slice)",
            marker_color="#2ca02c", opacity=0.55)
fig.add_scatter(x=pe.date, y=pe.PX_LAST, name="ICLN price ($)",
                line=dict(color="#111", width=2.5), secondary_y=True)
# For contrast, overlay the section-4 SHARE index (rescaled): it ramps, the count does not.
_share = g["share_smooth"].reindex(clean_ts.index, method="nearest")
_scale = (clean_ts.new_docs.max() or 1) / (_share.max() or 1)
fig.add_scatter(x=_share.index, y=_share * _scale, name="clean-energy SHARE (§4, rescaled)",
                line=dict(color="#d62728", width=2, dash="dot"))
fig.add_vline(x=PRICE_PEAK, line=dict(color="red", dash="dash"))
fig.add_annotation(x=PRICE_PEAK, yref="paper", y=1.0, yanchor="bottom", showarrow=False,
                   text="price ATH", font=dict(color="red"))
fig.update_layout(title="Unsupervised BERTrend clean-energy theme (count-based popularity) vs ICLN price",
                  template="plotly_white", height=460, legend=dict(orientation="h", y=1.12))
fig.update_yaxes(title_text="docs / slice (count)", secondary_y=False)
fig.update_yaxes(title_text="ICLN price ($)", secondary_y=True)
fig.write_html(OUTPUT_DIR / "bertrend_clean_energy_leadlag.html")
fig.show()

In [10]:
# --- "Detect in advance": walk the weak/strong signal classification forward in time ---
# At each snapshot date BERTrend buckets themes into noise / weak / strong using the
# trailing-window popularity percentiles. We check WHEN a clean-energy theme first shows up
# as a weak (emerging) or strong signal — and compare to the 2021-01-07 price peak.
keys = sorted(pd.Timestamp(k) for k in bertrend.doc_groups.keys())
clean_set = set(clean_ids)

snapshots = []
for k in keys:
    if not (pd.Timestamp("2020-01-01") <= k <= pd.Timestamp("2021-03-01")):
        continue
    _, w, s = bertrend.classify_signals(WINDOW_SIZE, k)
    def ids(d):
        return sorted(clean_set & (set(d["Topic"].astype(int))
                      if d is not None and not d.empty and "Topic" in d.columns else set()))
    snapshots.append((k, ids(w), ids(s)))

print(f"ICLN price/volume peak: {PRICE_PEAK.date()}\n")
print(f"{'snapshot':<13}{'clean WEAK (emerging)':<26}{'clean STRONG':<16}")
for k, wk, st in snapshots:
    print(f"{str(k.date()):<13}{str(wk) if wk else '-':<26}{str(st) if st else '-':<16}")

pre = [(k, wk, st) for k, wk, st in snapshots if k <= PRICE_PEAK]
flagged_pre = [k for k, wk, st in pre if wk or st]
if flagged_pre:
    first = min(flagged_pre)
    frac = len(flagged_pre) / len(pre)
    labels = "; ".join(f"T{t}: {rep_map.get(t, '')[:40]}" for t in clean_ids)
    print(f"\nUNSUPERVISED RESULT: with NO keyword filter on the corpus, BERTrend discovered")
    print(f"and named a clean-energy theme -> {labels}")
    print(f"It sits on the weak/strong signal board in {frac:.0%} of pre-peak snapshots,")
    print(f"first appearing {first.date()} — ~{(PRICE_PEAK - first).days // 30} months before the {PRICE_PEAK.date()} peak.")
    print("\nCAVEAT 1 (isolation): discovery is robust, but auto-PICKING this theme out of 228 is")
    print("the fragile step — at ~0.5% prevalence + anisotropic embeddings, it took a 3-way")
    print("clean/fossil/generic contrast to separate solar from boilerplate (see ranking above).")
    print("CAVEAT 2 (count vs share): BERTrend popularity is doc-COUNT based, but the detectable")
    print("clean-energy signal in §4 is a SHARE rise. So use BERTrend to *discover & name* the")
    print("theme unsupervised, then track it with a share-normalized intensity (§2-4) for timing.")
else:
    print("\n=> No clean-energy theme reached weak/strong before the peak at this configuration.")

ICLN price/volume peak: 2021-01-07

snapshot     clean WEAK (emerging)     clean STRONG    
2020-01-14   [86]                      -               
2020-01-28   -                         -               
2020-02-11   -                         -               
2020-02-25   -                         -               
2020-03-10   [86]                      -               
2020-03-24   [86]                      -               
2020-04-07   [86]                      -               
2020-04-21   -                         -               
2020-05-05   -                         -               
2020-05-19   -                         -               
2020-06-02   [86]                      -               
2020-06-16   [86]                      -               
2020-06-30   [86]                      -               
2020-07-14   [86]                      -               
2020-07-28   [86]                      -               
2020-08-11   [86]                      -               
2020-08-25  

### Unsupervised vs keyword — what BERTrend actually adds (and its limit here)

**What it adds:** with **no keyword filter on the corpus**, BERTrend *discovers* and *names* a clean-energy theme straight from the raw news — a coherent solar cluster (`solar, energy, plants, systems…`), labelled for free by c-TF-IDF. It is merged across slices, classified on the weak–strong board, and (per the stats above) is visible months before ICLN's 2021-01-07 peak. The reference centroids only *rank/label* the discovered clusters; they never filter the corpus or steer clustering.

**The honest limits.** (1) **Isolation is the fragile step, not discovery.** At ~0.5% prevalence with anisotropic embeddings, a plain clean-vs-fossil score ranks boilerplate (circuit-breakers, AGMs, "billionaire") as high as solar. Cleanly picking out the real theme needed mean-centering *and* a three-way `clean − max(fossil, generic)` contrast. (2) **Count vs share.** BERTrend popularity is a doc *count*; the detectable clean-energy signal in §2–4 is a *share* rise. Total Bloomberg volume *fell* into 2021, so the count (green bars) stays flatter than the share (red dotted line).

**Practical takeaway:** use BERTrend to **discover and name** themes with no lexicon, then **track the chosen theme with a share-normalised intensity** (§2–4) for clean lead-lag timing. UMAP/HDBSCAN are stochastic and slice-dependent; per-slice models and merged-theme state are persisted under `output/bertrend_clean_energy_models/` (and scored themes in `output/bertrend_theme_scores.parquet`). Fix seeds and validate weak→strong thresholds on held-out themes for production.

## 7. How does the news–price correlation evolve over time?

The lead–lag plots show *levels*; here we ask how the **relationship** itself moves through time. We compute a **rolling (trailing-window) Pearson correlation** of each news signal against ICLN price, on two views:

- **Levels** — do the news signal and the price *trend together*?
- **Changes** (period-over-period diffs) — do they *co-move* bi-weekly?

Two signals are tracked: the §2–4 keyword **SHARE** index and the §6 unsupervised **BERTrend theme** popularity. Vertical markers show the regime onset, the real-time trigger, and the 2021-01-07 price peak, so you can read how the correlation behaves *around* those dates.

> **Read with care.** Over a short window two *trending* series correlate near ±1 almost mechanically — the magnitude is inflated and only the sign (their local co-trend direction) really moves. To keep it honest we use a longer (~5-month) window, **shade the band where |ρ| is not statistically significant** (p>0.05 for the window length), and show the **change-based** panel, where differencing removes the trend and the inflation collapses. Trust sustained, significant moves (e.g. the lock-in into the peak and the post-peak breakdown), not the noisy early whipsaw.

In [11]:
# --- Rolling correlation of each news signal vs ICLN price, through time ---
# Levels of two trending series inflate |rho| toward 1 over short windows, and the sign just
# tracks their local slope. So we (i) use a longer, more stable window, (ii) shade the band
# where |rho| is NOT significant (p>0.05), and (iii) add a change-based panel (differencing
# removes the trend artifact) as the honest cross-check.
from scipy import stats

CORR_WIN = 10                       # bi-weekly slices (~5 months) — longer = more stable
_t = stats.t.ppf(0.975, CORR_WIN - 2)
RCRIT = float(_t / np.sqrt(_t**2 + (CORR_WIN - 2)))   # 5% two-sided critical |rho| for n=CORR_WIN

cf = pd.DataFrame(index=m.index)
cf["price"] = m["PX_LAST"]
cf["share"] = m["share_smooth"]                                      # §2-4 keyword SHARE
cf["theme"] = (clean_ts["new_docs"].reindex(m.index).fillna(0.0)     # §6 BERTrend popularity
               .rolling(3, min_periods=1).mean())
cf = cf.loc[DATE_START:DATE_END]

lev = pd.DataFrame({"share": cf["share"].rolling(CORR_WIN).corr(cf["price"]),
                    "theme": cf["theme"].rolling(CORR_WIN).corr(cf["price"])})
chg = cf.diff()
mov = pd.DataFrame({"share": chg["share"].rolling(CORR_WIN).corr(chg["price"]),
                    "theme": chg["theme"].rolling(CORR_WIN).corr(chg["price"])})
pd.concat({"level": lev, "change": mov}, axis=1).to_parquet(OUTPUT_DIR / "clean_energy_rolling_corr.parquet")

RED, GREEN = "#d62728", "#2ca02c"
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.13,
                    subplot_titles=("Levels — trend co-movement (|ρ| inflated; read with care)",
                                    "Changes — period-to-period co-movement (trend removed)"))
for r, src in [(1, lev), (2, mov)]:
    fig.add_hrect(y0=-RCRIT, y1=RCRIT, line_width=0, fillcolor="#9aa0a6", opacity=0.16, row=r, col=1)
    fig.add_hline(y=0, line=dict(color="#cccccc", width=1), row=r, col=1)
    fig.add_scatter(x=src.index, y=src["share"], name="keyword SHARE vs price",
                    line=dict(color=RED, width=2.5), row=r, col=1,
                    legendgroup="s", showlegend=(r == 1))
    fig.add_scatter(x=src.index, y=src["theme"], name="BERTrend theme vs price",
                    line=dict(color=GREEN, width=2.5), row=r, col=1,
                    legendgroup="t", showlegend=(r == 1))

# Event markers: dotted vlines on both panels + a compact, color-coded legend in the top-left
# corner (avoids the label pile-up when the trigger and the peak nearly coincide).
events = sorted([e for e in [(onset, "#1f77b4", "regime onset"),
                             (detection, "#0a8f3c", "real-time trigger"),
                             (PRICE_PEAK, "#111111", "price ATH")] if e[0] is not None],
                key=lambda e: e[0])
for dte, col, _txt in events:
    fig.add_vline(x=dte, line=dict(color=col, dash="dot", width=1.5), row="all", col=1)
for i, (dte, col, txt) in enumerate(events):
    fig.add_annotation(xref="paper", yref="paper", x=0.012, y=0.985 - 0.055 * i,
                       xanchor="left", yanchor="top", showarrow=False,
                       text=f"<b>┊</b> {txt} · {dte.date()}", font=dict(color=col, size=11),
                       bgcolor="rgba(255,255,255,0.65)")
fig.add_annotation(xref="paper", yref="paper", x=0.99, y=0.985, xanchor="right", yanchor="top",
                   showarrow=False, text=f"grey band: |ρ| < {RCRIT:.2f} not significant (p>0.05, n={CORR_WIN})",
                   font=dict(color="#666", size=10))

fig.update_yaxes(range=[-1.05, 1.05], title_text="ρ — levels", row=1, col=1)
fig.update_yaxes(range=[-1.05, 1.05], title_text="ρ — changes", row=2, col=1)
fig.update_layout(title=f"Rolling news–price correlation ({CORR_WIN}-slice ≈ {CORR_WIN * GRANULARITY_DAYS // 7}-week trailing window)",
                  template="plotly_white", height=620, margin=dict(t=70, b=70),
                  legend=dict(orientation="h", yanchor="top", y=-0.12, x=0.5, xanchor="center"))
fig.write_html(OUTPUT_DIR / "clean_energy_rolling_corr.html")
fig.show()

# Numeric read of the LEVEL correlation around the key dates
def _at(s, dte):
    return float(s.reindex([dte], method="nearest").iloc[0])
post = PRICE_PEAK + pd.Timedelta(days=90)
if onset is not None:
    print(f"LEVEL corr(SHARE, price):  onset {_at(lev['share'], onset):+.2f}  ->  peak {_at(lev['share'], PRICE_PEAK):+.2f}  ->  +3m {_at(lev['share'], post):+.2f}")
    print(f"LEVEL corr(THEME, price):  onset {_at(lev['theme'], onset):+.2f}  ->  peak {_at(lev['theme'], PRICE_PEAK):+.2f}  ->  +3m {_at(lev['theme'], post):+.2f}")
    print(f"(|ρ| below {RCRIT:.2f} is not significant at n={CORR_WIN}; ignore the shaded band.)")
    print("\nThe SHARE–price correlation tightens into the peak (both rising), then breaks down")
    print("afterwards as price falls while news stays elevated. The count-based BERTrend theme")
    print("correlation is noisier and only co-trends briefly around the peak.")

LEVEL corr(SHARE, price):  onset +0.76  ->  peak +0.97  ->  +3m +0.42
LEVEL corr(THEME, price):  onset -0.30  ->  peak +0.20  ->  +3m -0.67
(|ρ| below 0.63 is not significant at n=10; ignore the shaded band.)

The SHARE–price correlation tightens into the peak (both rising), then breaks down
afterwards as price falls while news stays elevated. The count-based BERTrend theme
correlation is noisier and only co-trends briefly around the peak.


In [2]:
# --- Same correlations as |ρ| (magnitude of association, sign ignored) ---
# Self-contained: reads the saved series, so it does NOT need §1-6 re-run (no reload, no BERTrend).
corr = pd.read_parquet(OUTPUT_DIR / "clean_energy_rolling_corr.parquet")
levA, movA = corr["level"].abs(), corr["change"].abs()

# Window + significance threshold and event dates: reuse §7 if in memory, else fall back.
try:
    _win, _rc = CORR_WIN, RCRIT
except NameError:
    from scipy import stats
    _win = 10; _t = stats.t.ppf(0.975, _win - 2); _rc = float(_t / np.sqrt(_t**2 + (_win - 2)))
try:
    _ev = [(onset, "#1f77b4", "regime onset"), (detection, "#0a8f3c", "real-time trigger"),
           (PRICE_PEAK, "#111111", "price ATH")]
except NameError:  # standalone: recompute the §3-4 markers from the saved lead-lag table
    _mm = pd.read_parquet(OUTPUT_DIR / "clean_energy_lead_lag.parquet")
    _bl = _mm.loc[DATE_START:pd.Timestamp("2020-06-30"), "share_smooth"].median()
    _ix, _ab, _det = _mm.index, _mm["share_smooth"] > 1.5 * _bl, None
    for k in range(len(_mm) - 1):
        if _ix[k] > pd.Timestamp("2020-06-30") and _ab.iloc[k] and _ab.iloc[k + 1]:
            _det = _ix[k]; break
    _ons, _sm = None, _mm["share_smooth"]
    for k in range(len(_mm)):
        if bool((_sm.iloc[k:] >= 1.2 * _bl).all()):
            _ons = _ix[k]; break
    _ev = [(_ons, "#1f77b4", "regime onset"), (_det, "#0a8f3c", "real-time trigger"),
           (_mm["PX_LAST"].idxmax(), "#111111", "price ATH")]
_events = sorted([e for e in _ev if e[0] is not None], key=lambda e: e[0])

RED, GREEN = "#d62728", "#2ca02c"
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.13,
                    subplot_titles=("|ρ| — levels (trend co-movement)", "|ρ| — changes (trend removed)"))
for r, src in [(1, levA), (2, movA)]:
    fig.add_hrect(y0=0, y1=_rc, line_width=0, fillcolor="#9aa0a6", opacity=0.16, row=r, col=1)
    fig.add_hline(y=_rc, line=dict(color="#888", dash="dash", width=1), row=r, col=1)
    fig.add_scatter(x=src.index, y=src["share"], name="keyword SHARE vs price",
                    line=dict(color=RED, width=2.5), row=r, col=1, legendgroup="s", showlegend=(r == 1))
    fig.add_scatter(x=src.index, y=src["theme"], name="BERTrend theme vs price",
                    line=dict(color=GREEN, width=2.5), row=r, col=1, legendgroup="t", showlegend=(r == 1))
for dte, col, _txt in _events:
    fig.add_vline(x=dte, line=dict(color=col, dash="dot", width=1.5), row="all", col=1)
for i, (dte, col, txt) in enumerate(_events):
    fig.add_annotation(xref="paper", yref="paper", x=0.012, y=0.985 - 0.055 * i, xanchor="left",
                       yanchor="top", showarrow=False, text=f"<b>┊</b> {txt} · {dte.date()}",
                       font=dict(color=col, size=11), bgcolor="rgba(255,255,255,0.65)")
fig.add_annotation(xref="paper", yref="paper", x=0.99, y=0.985, xanchor="right", yanchor="top",
                   showarrow=False, text=f"grey band: |ρ| < {_rc:.2f} not significant (p>0.05, n={_win})",
                   font=dict(color="#666", size=10))
fig.update_yaxes(range=[0, 1.05], title_text="|ρ| — levels", row=1, col=1)
fig.update_yaxes(range=[0, 1.05], title_text="|ρ| — changes", row=2, col=1)
fig.update_layout(title=f"Absolute rolling news–price correlation |ρ| ({_win}-slice ≈ {_win * 2}-week window)",
                  template="plotly_white", height=620, margin=dict(t=70, b=70),
                  legend=dict(orientation="h", yanchor="top", y=-0.12, x=0.5, xanchor="center"))
fig.write_html(OUTPUT_DIR / "clean_energy_rolling_corr_abs.html")
fig.show()